# Qwen3.5-27B QLoRA on an M5 Mac (MLX)

This notebook fine-tunes the 4-bit `mlx-community/Qwen3.5-27B-4bit` checkpoint for rubric grading on a 48 GB M5 Mac. It uses MLX/Metal instead of CUDA, masks the prompt so loss is calculated only on the assistant JSON answer, and verifies the Mac's unified-memory capacity.

Requirements: Apple Silicon, macOS 14 or newer, Python 3.11 or 3.12, at least 32 GB unified memory, roughly 25 GB free disk space, and `training_data.csv` beside this notebook. Close memory-heavy applications before training. The initial model download is approximately 16 GB.

Run the cells in order. Training is deliberately gated behind `RUN_TRAINING = False` so Run All cannot accidentally start a multi-hour job.

## 1. Install the native Apple Silicon stack

Use a native arm64 Python environment, not a Rosetta/x86 environment. Restart the kernel after this cell if Jupyter requests it.

In [ ]:
%pip install -U "mlx-lm[train]>=0.30.7" pandas pyyaml transformers

## 2. Hardware check and memory-aware profile

The 27B checkpoint itself occupies about 16 GB. The profile changes only the number of final transformer layers receiving adapters and the LoRA rank; batch size remains one. Override values only after a successful short run.

In [ ]:
import ast
import csv
import json
import math
import platform
import re
import subprocess
import sys
from pathlib import Path

import mlx.core as mx
import pandas as pd
import yaml

if platform.system() != "Darwin" or platform.machine() != "arm64":
    raise RuntimeError("This notebook must run in a native arm64 Python environment on Apple Silicon.")
if not mx.metal.is_available():
    raise RuntimeError("MLX cannot access Metal. Check the Python architecture and macOS installation.")

memory_bytes = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"], text=True).strip())
MEMORY_GB = round(memory_bytes / 1024**3)

if MEMORY_GB < 32:
    raise MemoryError(
        f"Detected {MEMORY_GB} GB. Qwen3.5-27B QLoRA is not safe at this capacity. "
        "Use mlx-community/Qwen3.5-9B-4bit instead."
    )
elif MEMORY_GB < 48:
    PROFILE = {"name": "32 GB conservative", "num_layers": 4, "rank": 4, "grad_checkpoint": True}
elif MEMORY_GB < 64:
    PROFILE = {"name": "48 GB balanced", "num_layers": 8, "rank": 8, "grad_checkpoint": False}
elif MEMORY_GB < 96:
    PROFILE = {"name": "64 GB balanced", "num_layers": 12, "rank": 8, "grad_checkpoint": False}
else:
    PROFILE = {"name": "96+ GB quality", "num_layers": 16, "rank": 16, "grad_checkpoint": False}

print(f"Python: {sys.version.split()[0]} ({platform.machine()})")
print(f"MLX: {mx.__version__}; Metal available: {mx.metal.is_available()}")
print(f"Unified memory: {MEMORY_GB} GB")
print("Selected profile:", PROFILE)

## 3. Experiment configuration

Two epochs are a practical first run for this 2,165-row dataset. MLX counts micro-batch iterations, so the exact iteration count is calculated after splitting the data.

In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-27B-4bit"
DATASET_PATH = Path("./training_data.csv")
DATA_DIR = Path("./mlx-qwen35-27b-data")
OUTPUT_DIR = Path("./qwen35-27b-grading-mlx-lora")
CONFIG_PATH = Path("./qwen35-27b-m5-lora.yaml")

SEED = 42
NUM_EPOCHS = 2
MAX_SEQ_LENGTH = 448
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-5
LORA_DROPOUT = 0.05

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Place training_data.csv at {DATASET_PATH.resolve()}")

## 4. Load and validate the grading data

In [ ]:
df = pd.read_csv(DATASET_PATH)
required = ["question", "desired_answer", "student_answer", "criteria[]", "criteria_score[]"]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def parse_array(value):
    """Parse JSON/Python lists and the dataset's unquoted bracket format."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value).strip()
    for _ in range(2):
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
            continue
        break
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            text = parsed.strip()
    except (ValueError, SyntaxError):
        pass
    if not (text.startswith("[") and text.endswith("]")):
        raise ValueError(f"Expected a bracketed list, got {value!r}")
    inner = text[1:-1].strip()
    if not inner:
        return []
    if re.search(r",(?=Answer\s)", inner):
        return [item.strip() for item in re.split(r",(?=Answer\s)", inner)]
    return [item.strip() for item in next(csv.reader([inner], skipinitialspace=True))]

def parse_criteria(value, expected_count):
    items = parse_array(value)
    if len(items) == expected_count:
        return items
    text = str(value).strip()
    for _ in range(2):
        try:
            decoded = json.loads(text)
        except json.JSONDecodeError:
            break
        if isinstance(decoded, str):
            text = decoded.strip()
        else:
            break
    inner = text[1:-1].strip() if text.startswith("[") and text.endswith("]") else text
    items = [inner] if expected_count == 1 else re.split(r",(?=Answer\s)", inner)
    items = [item.strip() for item in items]
    if len(items) != expected_count:
        raise ValueError(f"Parsed {len(items)} criteria but expected {expected_count}: {value!r}")
    return items

df["criteria_score"] = df["criteria_score[]"].apply(lambda v: [int(x) for x in parse_array(v)])
df["criteria"] = df.apply(lambda r: parse_criteria(r["criteria[]"], len(r["criteria_score"])), axis=1)

bad_rows = []
for index, row in df.iterrows():
    if not row["criteria"] or len(row["criteria"]) != len(row["criteria_score"]):
        bad_rows.append((index, "criterion/label length mismatch"))
    elif any(int(score) not in (0, 1) for score in row["criteria_score"]):
        bad_rows.append((index, "non-binary label"))
if bad_rows:
    raise ValueError(f"Invalid rows (first 10): {bad_rows[:10]}")
print(f"Validated {len(df):,} rows")
df[["criteria", "criteria_score"]].head()

## 5. Build assistant-only chat examples and deterministic splits

The final chat message is the completion. `mask_prompt: true` in the MLX configuration excludes the system and user tokens from loss. Grading totals and maximum scores never enter the prompt.

In [ ]:
SYSTEM_PROMPT = """You are a strict grading assistant.

Evaluate whether a student's answer fulfills each grading criterion independently.
For every criterion, output 1 if it is fulfilled and 0 if it is not.
Return ONLY a valid JSON array containing exactly one 0 or 1 per criterion, in the original order.
Do not include explanations, markdown, or any text outside the JSON array."""

def make_example(row):
    criteria_text = "\n".join(f"{i + 1}. {criterion}" for i, criterion in enumerate(row["criteria"]))
    user_prompt = (
        f"Question:\n{row['question']}\n\n"
        f"Desired answer:\n{row['desired_answer']}\n\n"
        f"Student answer:\n{row['student_answer']}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    target = json.dumps([int(x) for x in row["criteria_score"]], separators=(",", ":"))
    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]},
        ]
    }

examples = [make_example(row) for _, row in df.iterrows()]
shuffled = df.sample(frac=1, random_state=SEED).index.tolist()
n_train = int(len(shuffled) * 0.80)
n_valid = int(len(shuffled) * 0.10)
split_indices = {
    "train": shuffled[:n_train],
    "valid": shuffled[n_train:n_train + n_valid],
    "test": shuffled[n_train + n_valid:],
}
splits = {name: [make_example(df.loc[i]) for i in indices] for name, indices in split_indices.items()}
print({name: len(rows) for name, rows in splits.items()})
print(json.dumps(splits["train"][0], indent=2, ensure_ascii=False))

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, rows in splits.items():
    path = DATA_DIR / f"{split_name}.jsonl"
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote {len(rows):,} examples to {path}")

## 6. Verify sequence lengths

This downloads tokenizer files only. The cell fails rather than silently truncating training examples. If your CSV changes and the maximum exceeds 448, increase the cap cautiously because activation memory grows with sequence length.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def formatted_token_count(example):
    kwargs = {"tokenize": True, "add_generation_prompt": False}
    try:
        ids = tokenizer.apply_chat_template(example["messages"], enable_thinking=False, **kwargs)
    except TypeError:
        ids = tokenizer.apply_chat_template(example["messages"], **kwargs)
    return len(ids)

lengths = [formatted_token_count(example) for example in examples]
print(pd.Series(lengths).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
if max(lengths) > MAX_SEQ_LENGTH:
    raise ValueError(
        f"Longest example is {max(lengths)} tokens, above MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
        "Increase the cap or shorten the examples; do not train with silent truncation."
    )
print("All examples fit. Longest sequence:", max(lengths))

## 7. Write the optimized MLX-LM configuration

Adapters target both Qwen3.5 attention families: Q/K/V/O projections in full-attention layers and QKV/Z/output projections in linear-attention layers. MLP adapters are omitted to control memory and training time.

In [ ]:
train_micro_batches_per_epoch = math.ceil(len(splits["train"]) / BATCH_SIZE)
TOTAL_ITERS = train_micro_batches_per_epoch * NUM_EPOCHS
STEPS_PER_EVAL = max(50, train_micro_batches_per_epoch // 4)
SAVE_EVERY = max(100, train_micro_batches_per_epoch // 2)

config = {
    "model": MODEL_ID,
    "train": True,
    "fine_tune_type": "lora",
    "optimizer": "adamw",
    "optimizer_config": {"adamw": {"weight_decay": 0.01}},
    "data": str(DATA_DIR),
    "seed": SEED,
    "num_layers": PROFILE["num_layers"],
    "batch_size": BATCH_SIZE,
    "grad_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "iters": TOTAL_ITERS,
    "val_batches": min(32, len(splits["valid"])),
    "learning_rate": LEARNING_RATE,
    "steps_per_report": 10,
    "steps_per_eval": STEPS_PER_EVAL,
    "save_every": SAVE_EVERY,
    "adapter_path": str(OUTPUT_DIR),
    "max_seq_length": MAX_SEQ_LENGTH,
    "grad_checkpoint": PROFILE["grad_checkpoint"],
    "clear_cache_threshold": 2 * 1024**3,
    "mask_prompt": True,
    "lora_parameters": {
        "keys": [
            "self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
            "linear_attn.in_proj_qkv", "linear_attn.in_proj_z", "linear_attn.out_proj",
        ],
        "rank": PROFILE["rank"],
        "scale": float(PROFILE["rank"] * 2),
        "dropout": LORA_DROPOUT,
    },
    "lr_schedule": {
        "name": "cosine_decay",
        "warmup": max(20, TOTAL_ITERS // 20),
        "warmup_init": 1e-7,
        "arguments": [LEARNING_RATE, TOTAL_ITERS, 5e-7],
    },
}

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(CONFIG_PATH.read_text())
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUMULATION_STEPS}")
print(f"Iterations: {TOTAL_ITERS:,} ({NUM_EPOCHS} passes over training data)")

## 8. Train

First set `RUN_TRAINING = True`. Keep the Mac connected to power, close large applications, and prevent sleep. The model downloads on the first run. MLX saves periodic checkpoints in the adapter directory.

If Metal reports an out-of-memory error, reduce `num_layers` by half. If a long Qwen3.5 run reports a Metal resource-limit error, restart Python and resume from the latest saved `adapters.safetensors` by adding `resume_adapter_file` to the YAML configuration.

In [ ]:
RUN_TRAINING = False  # Change to True when the configuration above looks correct.

command = [sys.executable, "-m", "mlx_lm", "lora", "--config", str(CONFIG_PATH)]
print("Command:", " ".join(command))
if RUN_TRAINING:
    subprocess.run(command, check=True)
else:
    print("Training not started. Set RUN_TRAINING=True and rerun this cell.")

## 9. Load the adapter and run deterministic inference

Run this only after training has produced `adapters.safetensors`. The model and adapter are loaded once and reused.

In [ ]:
from mlx_lm import generate, load
from mlx_lm.sample_utils import make_sampler

adapter_file = OUTPUT_DIR / "adapters.safetensors"
if not adapter_file.exists():
    raise FileNotFoundError(f"Train first; adapter not found at {adapter_file}")

model, mlx_tokenizer = load(MODEL_ID, adapter_path=str(OUTPUT_DIR))
greedy_sampler = make_sampler(temp=0.0)

def grading_messages(question, desired_answer, student_answer, criteria):
    criteria_text = "\n".join(f"{i + 1}. {criterion}" for i, criterion in enumerate(criteria))
    user = (
        f"Question:\n{question}\n\nDesired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\nCriteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": user}]},
    ]

def predict_scores(question, desired_answer, student_answer, criteria):
    messages = grading_messages(question, desired_answer, student_answer, criteria)
    kwargs = {"add_generation_prompt": True}
    try:
        prompt = mlx_tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        prompt = mlx_tokenizer.apply_chat_template(messages, **kwargs)
    raw = generate(
        model, mlx_tokenizer, prompt=prompt, max_tokens=max(32, len(criteria) * 6),
        sampler=greedy_sampler, verbose=False,
    ).strip()
    candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw)
    scores = json.loads(candidates[-1]) if candidates else None
    valid = isinstance(scores, list) and len(scores) == len(criteria) and all(x in (0, 1) for x in scores)
    return {"valid": valid, "scores": scores if valid else None, "raw_output": raw}

predict_scores(
    question="Explain why HTTPS is more secure than HTTP.",
    desired_answer="HTTPS encrypts network traffic using TLS.",
    student_answer="HTTPS protects transmitted data using encryption.",
    criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
)

## 10. Evaluate held-out examples

The default limit keeps evaluation practical on a laptop. Set `EVAL_LIMIT = None` for the complete test set. Invalid model responses count as incorrect for exact-match accuracy.

In [ ]:
EVAL_LIMIT = 50
test_indices = split_indices["test"] if EVAL_LIMIT is None else split_indices["test"][:EVAL_LIMIT]
records = []
flat_true, flat_pred = [], []

for row_index in test_indices:
    row = df.loc[row_index]
    truth = [int(x) for x in row["criteria_score"]]
    prediction = predict_scores(
        str(row["question"]), str(row["desired_answer"]),
        str(row["student_answer"]), row["criteria"],
    )
    exact = bool(prediction["valid"] and prediction["scores"] == truth)
    if prediction["valid"]:
        flat_true.extend(truth)
        flat_pred.extend(prediction["scores"])
    records.append({
        "row_index": int(row_index), "truth": truth,
        "prediction": prediction["scores"] if prediction["valid"] else prediction["raw_output"],
        "valid": prediction["valid"], "exact_match": exact,
    })

evaluation_df = pd.DataFrame(records)
tp = sum(t == 1 and p == 1 for t, p in zip(flat_true, flat_pred))
fp = sum(t == 0 and p == 1 for t, p in zip(flat_true, flat_pred))
fn = sum(t == 1 and p == 0 for t, p in zip(flat_true, flat_pred))
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
metrics = {
    "examples": len(evaluation_df),
    "invalid_output_rate": 1 - evaluation_df["valid"].mean() if len(evaluation_df) else 0.0,
    "exact_match_accuracy": evaluation_df["exact_match"].mean() if len(evaluation_df) else 0.0,
    "per_criterion_accuracy_on_valid": sum(t == p for t, p in zip(flat_true, flat_pred)) / len(flat_true) if flat_true else 0.0,
    "precision_on_valid": precision,
    "recall_on_valid": recall,
    "f1_on_valid": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
}
display(pd.Series(metrics, name="fine_tuned_adapter"))
evaluation_df.head(20)

## Operational notes

- Activity Monitor's Memory Pressure is more useful than free-memory figures on macOS. Keep it green.
- A 32 GB Mac should retain the 4-layer/rank-4 profile. On 48 GB, try 8 layers before increasing rank.
- For repeatable comparisons, retain the generated JSONL split files and YAML configuration with the adapter.
- Do not fuse the adapter until evaluation is complete; keeping it separate is smaller and makes experiments easier to compare.
- The adapter is framework-specific. Export or fuse through MLX-LM if the final deployment format requires merged weights.